In [11]:
import os
import pandas as pd
from openai import OpenAI

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [13]:
# ---- 2. Load pairs and pick a small deliberate sample ----
pairs = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_pairs.csv")  # adjust path if needed

positives = pairs[pairs['label'] == 1].sample(5, random_state=1)
negatives = pairs[pairs['label'] == 0].sample(5, random_state=1)
sample = pd.concat([positives, negatives]).reset_index(drop=True)

In [14]:
# ---- 3. Zero-shot prompt function ----
def classify_pair(name_a, name_b):
    prompt = f"""You are comparing two product listings to determine if they refer to the same product.

Product A: {name_a}
Product B: {name_b}

Respond with only one word: MATCH or NO_MATCH."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,   # deterministic for now — we'll test higher temps later for consistency checks
        max_tokens=5
    )
    return response.choices[0].message.content.strip()

In [15]:
# ---- 4. Run on the sample ----
results = []
for _, row in sample.iterrows():
    raw_response = classify_pair(row['name_abt'], row['name_buy'])
    # Basic parsing — handle exact match first, fall back to keyword search
    if raw_response.upper() == "MATCH":
        pred_label = 1
    elif raw_response.upper() == "NO_MATCH":
        pred_label = 0
    elif "NO_MATCH" in raw_response.upper():
        pred_label = 0
    elif "MATCH" in raw_response.upper():
        pred_label = 1
    else:
        pred_label = None  # couldn't parse — flag it

    results.append({
        'name_abt': row['name_abt'],
        'name_buy': row['name_buy'],
        'true_label': row['label'],
        'raw_response': raw_response,
        'pred_label': pred_label
    })

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 50)
print(results_df)

                                            name_abt  \
0                     Weber Cast Iron Griddle - 7531   
1      Logitech Harmony RF Wireless Extender - RFEXT   
2    Sirius Plug And Play Universal Home Kit - SUPH1   
3  Sony 46' BRAVIA W-Series Black LCD Flat Panel ...   
4  Denon Blu-ray Disc DVD/CD Digital Player/Trans...   
5  Sony Black HD Radio With Dock For iPod And iPh...   
6  Panasonic DECT 6.0 Silver Digital Cordless Han...   
7     Pioneer KURO 50' Black Plasma HDTV - PDP5020FD   
8  Panasonic VIERA 32' Class Widescreen LCD HDTV ...   
9   Samsung 61' Black DLP Projection HDTV - HL61A650   

                                            name_buy  true_label raw_response  \
0  Weber Cast Iron Griddle for Genesis Silver As ...           1        MATCH   
1          Logitech Harmony RF Extender - 915-000044           1        MATCH   
2             Sirius SUPH1 Sirius Universal Home Kit           1        MATCH   
3  Sony BRAVIA W Series KDL-46W4100 46' LCD TV - ...       

In [16]:
# ---- 5. Quick check: how many parsed cleanly? ----
n_unparsed = results_df['pred_label'].isna().sum()
print(f"\nUnparsed responses: {n_unparsed} out of {len(results_df)}")


Unparsed responses: 0 out of 10


In [18]:
sample

,id_abt,id_buy,name_abt,name_buy,label
0,34256,206888001,Weber Cast Iron Griddle - 7531,Weber Cast Iron Griddle for Genesis Silver As ...,1
1,27182,206481908,Logitech Harmony RF Wireless Extender - RFEXT,Logitech Harmony RF Extender - 915-000044,1
2,32493,203324970,Sirius Plug And Play Universal Home Kit - SUPH1,Sirius SUPH1 Sirius Universal Home Kit,1
3,34894,208504340,Sony 46' BRAVIA W-Series Black LCD Flat Panel ...,Sony BRAVIA W Series KDL-46W4100 46' LCD TV - ...,1
4,34260,207390507,Denon Blu-ray Disc DVD/CD Digital Player/Trans...,Denon DVD-2500BTCI Blu-ray Disc Player,1
5,38773,202635701,Sony Black HD Radio With Dock For iPod And iPh...,Uniden TCX905 Cordless Handset - TCX905,0
6,35983,208154765,Panasonic DECT 6.0 Silver Digital Cordless Han...,Sony Cyber-shot DSC-W170 Digital Camera - Red ...,0
7,35276,202045862,Pioneer KURO 50' Black Plasma HDTV - PDP5020FD,Panasonic NN-H965BF Microwave Oven - Counter T...,0
8,35431,209026638,Panasonic VIERA 32' Class Widescreen LCD HDTV ...,Toshiba 52RV535U - 52' Widescreen 1080p LCD HD...,0
9,34136,208456233,Samsung 61' Black DLP Projection HDTV - HL61A650,Whirlpool WTW6700TW 28' Cabrio Series Top Load...,0
